In [1]:
import pandas as pd
import numpy as np
import random

# Configuración inicial
NUM_REGISTROS = 80000
SEMILLA = 42
np.random.seed(SEMILLA)
random.seed(SEMILLA)

# ---------------------------------------------------------
# 1. DEFINICIÓN DE DATOS MAESTROS (Contexto Dominicano)
# ---------------------------------------------------------

# Lista de 50+ Productos Agrícolas Comunes en RD
productos = [
    # Granos y Cereales
    "Arroz", "Maíz", "Habichuelas Rojas", "Habichuelas Negras", "Guandules", 
    
    # Musáceas
    "Plátano", "Guineo", "Rulo", 
    
    # Raíces y Tubérculos
    "Yuca", "Batata", "Ñame", "Yautía Coco", "Yautía Blanca", "Papa", "Mapuey",
    
    # Hortalizas y Vegetales
    "Tomate de Ensalada", "Tomate Industrial", "Ají Cubanela", "Ají Morrón", 
    "Cebolla Roja", "Ajo", "Repollo", "Lechuga", "Zanahoria", "Remolacha", 
    "Berenjena", "Molondrón", "Pepino", "Auyama", "Tayota", "Brócoli", 
    "Coliflor", "Vainita", "Cilantro",
    
    # Frutales
    "Aguacate", "Mango", "Chinola", "Lechosa", "Piña", "Limón Persa", 
    "Limón Agrio", "Naranja Dulce", "Naranja Agria", "Zapote", "Cereza", 
    "Melón", "Sandía", "Coco",
    
    # Agroindustriales
    "Cacao", "Café", "Tabaco", "Caña de Azúcar", "Maní"
]

# Regiones y sus características climáticas predominantes
regiones_info = {
    "Norte (Cibao)": ["Tropical Húmedo", "Templado (Montaña)"],
    "Sur": ["Semiárido", "Bosque Seco", "Templado (Sierra)"],
    "Este": ["Tropical de Sabana", "Tropical Húmedo"]
}

# Tipos de preparación de suelo
preparaciones_suelo = [
    "Arado Mecanizado", "Arado con Bueyes", "Corte y Quema (Tradicional)", 
    "Siembra Directa", "Nivelación Láser (Arroz)", "Bancales elevados"
]

# Mercados Mayoristas Principales en RD
mercados = [
    "Merca Santo Domingo", "Mercado Nuevo (D.N.)", "Hospedaje Yaque (Santiago)", 
    "Mercado de San Juan", "Mercado de la Vega"
]

# Años a simular
anios = [2020, 2021, 2022, 2023, 2024]

# Meses
meses = list(range(1, 13))

# ---------------------------------------------------------
# 2. GENERACIÓN DE DATOS
# ---------------------------------------------------------

data = []

for _ in range(NUM_REGISTROS):
    
    # Selección aleatoria básica
    region = random.choice(list(regiones_info.keys()))
    clima = random.choice(regiones_info[region])
    producto = random.choice(productos)
    anio = random.choice(anios)
    mes = random.choice(meses)
    mercado = random.choice(mercados)
    prep_suelo = random.choice(preparaciones_suelo)
    
    # --- Lógica Sintética (Para dar realismo) ---
    
    # 1. Área Sembrada (en Tareas): 
    # Pequeños productores (5-50 tareas), Medianos (50-500), Grandes (500+)
    # Usamos una distribución log-normal para simular que hay más pequeños que grandes
    area_tareas = int(np.random.lognormal(mean=3.5, sigma=1.0) * 5)
    if area_tareas < 1: area_tareas = 1
    
    # 2. Unidades de Medida y Rendimiento Base
    unidad = "Quintales (QQ)" # Estándar genérico
    rendimiento_base = 2.5 # QQ por tarea promedio
    
    if producto in ["Plátano", "Guineo", "Coco"]:
        unidad = "Millares"
        rendimiento_base = 0.3 # 300 unidades por tarea aprox por corte
    elif producto in ["Arroz", "Maíz"]:
        rendimiento_base = 4.5 # QQ por tarea
        if prep_suelo == "Nivelación Láser (Arroz)": rendimiento_base += 1.0
    elif producto in ["Ají Cubanela", "Tomate"]:
        unidad = "Libras"
        rendimiento_base = 2000 # Libras por tarea
    
    # Ajuste de rendimiento por clima (aleatorio controlado)
    factor_clima = np.random.uniform(0.7, 1.3) # Variación del 30%
    produccion_total = round(area_tareas * rendimiento_base * factor_clima, 2)
    
    # 3. Precio (Simulación de Oferta y Demanda)
    # Precio base aleatorio según producto (simplificado)
    precio_base = np.random.uniform(500, 3000) 
    
    if unidad == "Libras":
        precio_base = np.random.uniform(15, 60)
    elif unidad == "Millares":
        precio_base = np.random.uniform(3000, 8000)
        
    # Variación de precio por mes (estacionalidad simple)
    # Meses de sequía o alta demanda suben precio
    factor_mes = 1.0
    if mes in [6, 7, 12]: # Picos de demanda o clima
        factor_mes = 1.2
    
    precio_mercado = round(precio_base * factor_mes * np.random.uniform(0.9, 1.1), 2)
    
    # Agregar al dataset
    data.append([
        anio, 
        mes, 
        region, 
        clima, 
        producto, 
        prep_suelo, 
        area_tareas, 
        unidad, 
        produccion_total, 
        mercado, 
        precio_mercado
    ])

# Crear DataFrame
df = pd.DataFrame(data, columns=[
    "Año", "Mes", "Región", "Clima", "Producto", 
    "Preparación_Suelo", "Área_Sembrada_Tareas", 
    "Unidad_Medida", "Producción_Total", "Mercado_Destino", 
    "Precio_Unitario_DOP"
])

# ---------------------------------------------------------
# 3. EXPORTACIÓN
# ---------------------------------------------------------

print(f"Dataset generado con {len(df)} registros.")
print(df.head())

# Guardar a CSV
df.to_csv("DF_agricultura_RD.csv", index=False, encoding='utf-8')
print("Archivo 'DF_agricultura_RD.csv' guardado exitosamente.")

Dataset generado con 80000 registros.
    Año  Mes         Región               Clima        Producto  \
0  2022    4           Este  Tropical de Sabana            Maíz   
1  2024    2           Este  Tropical de Sabana          Zapote   
2  2021    4  Norte (Cibao)     Tropical Húmedo         Plátano   
3  2024    7  Norte (Cibao)     Tropical Húmedo           Melón   
4  2020    3           Este     Tropical Húmedo  Caña de Azúcar   

             Preparación_Suelo  Área_Sembrada_Tareas   Unidad_Medida  \
0             Arado con Bueyes                   272  Quintales (QQ)   
1              Siembra Directa                   144  Quintales (QQ)   
2     Nivelación Láser (Arroz)                   803        Millares   
3              Siembra Directa                   356  Quintales (QQ)   
4  Corte y Quema (Tradicional)                    12  Quintales (QQ)   

   Producción_Total       Mercado_Destino  Precio_Unitario_DOP  
0           1394.38  Mercado Nuevo (D.N.)              1859.2